# Category research — unified bin extracts

Steering doc: [`.ai/initiatives/category_sales_inventory_and_taxonomy.md`](../../../.ai/initiatives/category_sales_inventory_and_taxonomy.md)

**Phase 1:** Run extracts from `scripts/sql/unified_bin*_public.sql`, review counts and samples, then pickle approved DataFrames. Canonical taxonomy + AI categorization come later.

**Optional CSV review (same SQL as the notebook):** from repo root, run `python workspace/notebooks/category-research/ai_scripts/ai_execute_sql.py unified_bin1_public.sql unified_bin1` (and `unified_bin2` / `unified_bin3`). Results land in `workspace/notebooks/category-research/ai_scripts/output/` as `unified_bin*_1.csv` (gitignored).

**Setup:** Run Jupyter from the **repo root** (`ecothrift-dashboard`), with venv active and `pip install -r workspace/notebooks/_shared/requirements-notebooks.txt` (pandas).

## 1. Environment — Django + `cr` on path

In [1]:
import os
import sys
from pathlib import Path

# Walk up from notebook dir to repo root
nb_dir = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd().resolve()
REPO = nb_dir
while REPO != REPO.parent:
    if (REPO / "manage.py").is_file():
        break
    REPO = REPO.parent
else:
    raise RuntimeError("Could not find repo root (no manage.py found in parent dirs)")

os.chdir(REPO)
CR_ROOT = REPO / "workspace" / "notebooks" / "category-research"
sys.path.insert(0, str(CR_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "ecothrift.settings")

import django
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
print("Django OK:", django.get_version())
print("CWD:", Path.cwd())

Django OK: 5.2
CWD: E:\ecothrift-dashboard


## 2. Imports

In [2]:
from cr import (
    BIN_NAMES,
    CACHE_DIR,
    load_pickle,
    print_tier_a_review,
    run_extract,
    run_extract_all,
    save_pickle,
    sample_df,
    summarize,
    top_values,
)

## 3. Extract Bin 1

In [7]:
df1 = run_extract("bin1")
summarize(df1)
print_tier_a_review(df1)

E:\ecothrift-dashboard\workspace\notebooks\category-research\cr\extract.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, _pandas_db_connection(alias))


rows: 53185
Tier A missing rate:
  manifest_category: 0.1539
  vendor_name: 0.1534
  product_title: 0.0000
  product_brand: 0.0227
  manifest_retail_value: 0.1534
  item_retail_amt: 0.0000
Top 10 manifest_category:
  'KITCHEN_AND_DINING': 5723
  'HOME_DECOR': 4885
  'HOUSEHOLD_ESSENTIALS': 4415
  'TOYS': 3321
  'HARDWARE AND TOOLS': 1632
  'STORAGE': 1202
  'OUTDOOR_FURNITURE': 885
  'Outdoors': 778
  'BATHROOM_ACCESSORIES': 744
  'KIDS_FURNITURE': 641
Sample 5 rows (Tier A):
          manifest_category vendor_name                                                    product_title product_brand  manifest_retail_value  item_retail_amt
42974        Mass Skin Care      Amazon  Vanicream Moisturizing Lotion with Pump Fragrance Free 16 Ounce     Vanicream                  14.24            14.24
4305                                                                      8K HDMI 2-Port Switch - Silver       Philips                    NaN            29.99
4813                                      

In [8]:
sample_df(df1, 20)

,bin,row_key,item_id,sku,manifest_row_id,manifest_has_row,manifest_category,manifest_subcategory,manifest_description,manifest_retail_value,...,cart_line_id,quantity,unit_price,line_total,processing_completed_at,on_shelf_at,inventory_purchase_order_id,product_id,sold_at,sold_for
42974,bin1,48005,48005,ITM33RXAQN,30384.0,True,Mass Skin Care,Sun,Vanicream Moisturizing Lotion with Pump | Frag...,14.24,...,None,None,None,None,2025-11-06 16:49:34.343677+00:00,2026-01-17,131.0,30416,2025-11-22 16:21:26.245574+00:00,8.03
4305,bin1,5716,5716,ITM33YVLET,NaN,False,,,,NaN,...,None,None,None,None,2025-08-20 21:11:22.742195+00:00,2026-01-17,NaN,2030,2025-09-15 15:19:12.952615+00:00,14.99
4813,bin1,6224,6224,ITMUTRT3BT,NaN,False,,,,NaN,...,None,None,None,None,2025-08-20 22:15:49.733460+00:00,2026-01-17,NaN,2179,2025-12-05 21:39:04.035081+00:00,11.90
9281,bin1,12638,12638,ITM22A22LW,4673.0,True,HOUSEHOLD_ESSENTIALS,STEAM,SALAV LR-900 Retro Edition Fabric Shaver and L...,36.99,...,None,None,None,None,2025-09-04 14:23:26.376363+00:00,2026-01-17,38.0,6571,2025-09-14 23:54:06.858173+00:00,28.65
23685,bin1,27914,27914,ITMQV52PB6,15616.0,True,OUTDOOR_SPORTS,,"Gatorade Insulated Squeeze Bottle, 30 fl oz (Red)",14.97,...,None,None,None,None,2025-10-14 15:13:41.151289+00:00,2026-01-17,65.0,15436,2025-11-10 01:53:59.359390+00:00,9.71
42661,bin1,48336,48336,ITMQ823UXH,30715.0,True,Baby Care,Infant/Child Skin Care,Johnson's CottonTouch Newborn Baby Face and Bo...,10.79,...,None,None,None,None,2025-11-06 15:39:27.370703+00:00,2026-01-17,131.0,30747,2025-11-07 16:20:11.433697+00:00,6.45
12301,bin1,12050,12050,ITM2UV4JM7,5146.0,True,HOME_DECOR,ENTERAIN/SERVE,"7"" Marble Serving Tray with Brass Handles Warm...",19.99,...,None,None,None,None,2025-09-16 15:45:23.317522+00:00,2026-01-17,39.0,6303,2025-09-25 16:46:52.111799+00:00,15.72
36497,bin1,40211,40211,ITMBC4HB77,24400.0,True,BATHROOM_ACCESSORIES,RUGS,"20""x58"" Everyday Chenille Bath Runner White - ...",20.00,...,None,None,None,None,2025-10-29 16:37:18.112235+00:00,2026-01-17,120.0,24533,2025-11-15 23:43:10.235399+00:00,12.81
22274,bin1,25821,25821,ITMBA9GMEK,13196.0,True,HOUSEHOLD_ESSENTIALS,LIQUID,Arm & Hammer Sensitive Skin Plus Liquid Laundr...,13.99,...,None,None,None,None,2025-10-10 15:42:53.685697+00:00,2026-01-17,61.0,13677,2025-10-20 22:38:13.624380+00:00,9.96
3419,bin1,4830,4830,ITM0PQKN0M,NaN,False,,,,NaN,...,None,None,None,None,2025-08-20 19:42:56.600925+00:00,2026-01-17,NaN,1752,2025-10-20 22:36:47.223914+00:00,0.00


In [9]:
top_values(df1, "manifest_category", 15) if "manifest_category" in df1.columns else None

manifest_category
                        8183
KITCHEN_AND_DINING      5723
HOME_DECOR              4885
HOUSEHOLD_ESSENTIALS    4415
TOYS                    3321
HARDWARE AND TOOLS      1632
STORAGE                 1202
OUTDOOR_FURNITURE        885
Outdoors                 778
BATHROOM_ACCESSORIES     744
KIDS_FURNITURE           641
OFFICE_SUPPLIES          599
OUTDOOR_SPORTS           531
Team Sports              517
LIGHTING                 499
Name: count, dtype: int64

## 4. Extract Bin 2

In [3]:
df2 = run_extract("bin2")
summarize(df2)
print_tier_a_review(df2)

rows: 7307
Tier A missing rate:
  manifest_category: 0.1884
  vendor_name: 0.1883
  product_title: 0.0000
  product_brand: 0.0779
  manifest_retail_value: 0.1883
  item_retail_amt: 0.0000
Top 10 manifest_category:
  'KITCHEN_AND_DINING': 1197
  'HOME_DECOR': 499
  'HOUSEHOLD_ESSENTIALS': 405
  'HARDWARE AND TOOLS': 295
  'OUTDOOR_SPORTS': 252
  'BATHROOM_ACCESSORIES': 222
  'Outdoors': 154
  'MIXED_SMALL_APPLIANCES': 122
  'OUTDOOR_FURNITURE': 110
  'Infant/Preschool': 105
Sample 5 rows (Tier A):
          manifest_category vendor_name                                                                      product_title    product_brand  manifest_retail_value  item_retail_amt
7063     KITCHEN_AND_DINING      Target                                 4pk 8.5" Melamine Salad Plates (May be incomplete)  Room Essentials                  10.00            10.00
4553     KITCHEN_AND_DINING      Target   Cotton Striped Chair Pad Black/Natural Indoor Seat Cushion 15"x15"x2.1" OEKO-TEX        Threshol

E:\ecothrift-dashboard\workspace\notebooks\category-research\cr\extract.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, _pandas_db_connection(alias))


E:\ecothrift-dashboard\workspace\notebooks\category-research\cr\extract.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, _pandas_db_connection(alias))


In [5]:
sample_df(df2, 20).T

,7063,4553,6505,1918,5227,5576,3193,3114,1373,2609,7092,3805,4677,6611,6441,101,5168,6654,2526,6091
bin,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2,bin2
row_key,17022-47912,15872-44955,16782-47253,14841-42051,16167-45776,16330-46165,15304-43437,15269-43344,14632-41441,15057-42775,17037-47947,15550-44128,15927-45091,16832-47381,16756-47177,14207-40090,16135-45696,16852-47433,15027-42688,16584-46755
item_id,60883,59968,23958,60852,32599,39382,57856,41897,53817,25536,123456,58346,61476,39169,50242,49634,60165,31467,58931,61673
sku,ITMNZ5MEB4,ITM2QRZSEK,ITMD5UQX3Y,ITMCPS5VEU,ITMCJZ24WX,ITMKG8595U,ITML24G92R,ITMF9U2AU5,ITMSUPBL8M,ITMUVNWHQV,ITMPINKTAG,ITMLMGUR7A,ITMJDQZE7A,ITM89ES4XH,ITMN5HGUJL,ITMP38ZXLQ,ITMEG5U8DH,ITM9FVTJNW,ITMTRR6MGN,ITML9QAVN9
manifest_row_id,39112.0,38472.0,12458.0,39081.0,19972.0,NaN,NaN,24874.0,35444.0,13025.0,NaN,37883.0,NaN,NaN,32621.0,32013.0,38530.0,18840.0,38166.0,NaN
manifest_has_row,True,True,True,True,True,False,False,True,True,True,False,True,False,False,True,True,True,True,True,False
manifest_category,KITCHEN_AND_DINING,KITCHEN_AND_DINING,KITCHEN_AND_DINING,KITCHEN_AND_DINING,Outdoor & Sports Toys,,,BATHROOM_ACCESSORIES,Dolls Toys,HOUSEHOLD_ESSENTIALS,,OUTDOOR_SPORTS,,,"Hunting, Airsoft and Paintball",Music,KITCHEN_AND_DINING,Sports,KITCHEN_AND_DINING,
manifest_subcategory,SPRING INLINE SEAS,CHAIR PAD,NAPKINS,PILLOWFORT,Pool & Water,,,THR PERF TOWEL SET,Pretend Play Toys,LITTER LINERS,,SKATES,,,Hunting Accessories,Music,FRY PAN/SKILLETS,Sports,BEVERAGE BOTTLE,
manifest_description,"4pk 8.5"" Melamine Salad Plates - Room Essentia...",Cotton Striped Chair Pad Black/Natural - Thres...,4pk Cotton Slub with Tassel Trim Tied Fringe N...,Kids' 14oz 2pk Dragon Single Wall Recycled Str...,SwimSchool Swim Trainer Vest – Small/Medium Ma...,,,6pc Performance Plus Textured Dot Bath Towel S...,Casdon Dyson Cordless Vacuum Interactive & Off...,Cat Litter Liners - L - 20ct - up&up (Please b...,,Roller Derby Youth Adult Custom Fit Quad Rolle...,,,Pyramex I-Force Sporty Dual Pane Anti-Fog Gogg...,Powerslave[180g LP] [2015 Remaster],"6"" Enameled Cast Iron Skillet Light Gray - Fig...",MOSISO Small Sling Backpack Crossbody Sling Ba...,12oz 2pk Stacking Short Tumbler with Lid Yello...,
manifest_retail_value,10.0,12.0,12.0,10.0,14.99,NaN,NaN,35.0,23.99,9.99,NaN,49.99,NaN,NaN,14.96,27.38,12.0,11.99,10.0,NaN


## 5. Extract Bin 3

In [12]:
df3 = run_extract("bin3")
summarize(df3)
print_tier_a_review(df3)

rows: 8584
Tier A missing rate:
  manifest_category: 0.1902
  vendor_name: 0.1902
  product_title: 0.0000
  product_brand: 0.0226
  manifest_retail_value: 0.1902
  item_retail_amt: 0.0000
Top 10 manifest_category:
  'KITCHEN_AND_DINING': 1411
  'HOME_DECOR': 1108
  'HOUSEHOLD_ESSENTIALS': 501
  'HARDWARE AND TOOLS': 301
  'Athletic Sports Apparel': 226
  'ADULT BEDDING': 197
  'Team Sports': 140
  'MIXED_LOTS': 136
  'Book': 130
  'Novelty/Party Supplies/Costume & Dress Up': 124
Sample 5 rows (Tier A):
                    manifest_category vendor_name                                                                        product_title product_brand  manifest_retail_value  item_retail_amt
4952  Hair Care and Beauty Appliances      Amazon  Remington Shine Therapy Argan Oil Keratin Hot Rollers Set Professional Hair Curlers     Remington                  44.99            44.99
6701               HARDWARE AND TOOLS  Home Depot                                   Husky 1/2" Drive 1-1/4" Deep 6

E:\ecothrift-dashboard\workspace\notebooks\category-research\cr\extract.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, _pandas_db_connection(alias))


In [13]:
sample_df(df3, 20)

,bin,row_key,item_id,sku,manifest_row_id,manifest_has_row,manifest_category,manifest_subcategory,manifest_description,manifest_retail_value,...,cart_line_id,quantity,unit_price,line_total,processing_completed_at,on_shelf_at,inventory_purchase_order_id,product_id,sold_at,sold_for
4952,bin3,46913,46913,ITMLVFAQU7,29292.0,True,Hair Care and Beauty Appliances,Hair Setters,Remington Shine Therapy Argan Oil & Keratin Tr...,44.99,...,None,None,None,None,2025-11-06 16:35:36.720879+00:00,2026-01-17,131.0,29324,NaT,NaN
6701,bin3,29219,29219,ITMTUB46ZR,16767.0,True,HARDWARE AND TOOLS,Ratchets & Sockets,1/2 in. Drive 1-1/4 in. Deep 6-Point Impact So...,9.97,...,None,None,None,None,2025-10-13 19:53:03.917083+00:00,2026-01-17,66.0,16587,NaT,NaN
6742,bin3,59688,59688,ITMU4NQQ8K,38420.0,True,KITCHEN_AND_DINING,TABLE RUNNER,"108"" x 14"" Cotton Textured Runner Gray - Thres...",24.00,...,None,None,None,None,2025-11-30 01:03:23.138323+00:00,2026-01-17,159.0,39935,NaT,NaN
222,bin3,57847,57847,ITM2YJRQK9,NaN,False,,,,NaN,...,None,None,None,None,2025-11-11 16:49:02.286086+00:00,2026-01-17,153.0,38977,NaT,NaN
5521,bin3,59605,59605,ITMNVWXWNX,38401.0,True,KITCHEN_AND_DINING,TABLE RUNNER,"108"" x 14"" Cotton Textured Runner Gray - Thres...",24.00,...,None,None,None,None,2025-11-29 16:26:39.333194+00:00,2026-01-17,159.0,39916,NaT,NaN
8196,bin3,45679,45679,ITMYUDWRTA,28060.0,True,Hasbro,Infant/Preschool - Licensed Toys,Hasbro® Don’t Break The Ice Game,9.99,...,None,None,None,None,2025-10-30 18:58:11.883917+00:00,2026-01-17,129.0,28091,NaT,NaN
1084,bin3,56048,56048,ITM6E99JJD,36587.0,True,PERSONAL_CARE,HEALTH & BEAUTY AIDS,SEKKISEI CLEANSING OIL,19.99,...,None,None,None,None,2025-11-19 21:07:11.841869+00:00,2026-01-17,146.0,37341,NaT,NaN
2758,bin3,51292,51292,ITMD8NLD7J,NaN,False,,,,NaN,...,None,None,None,None,2025-11-03 17:01:13.574548+00:00,2026-01-17,137.0,33643,2025-11-19 18:42:20.598710+00:00,25.00
6460,bin3,52369,52369,ITMSQZY6KQ,34025.0,True,Team Sports,Soccer Training Aids,Franklin Sports 6' x 12' Replacement Net & Bun...,39.99,...,None,None,None,None,2025-11-14 16:23:50.094392+00:00,2026-01-17,141.0,34711,NaT,NaN
3459,bin3,28482,28482,ITMFJY3NXD,16060.0,True,KITCHEN AND BATH,,SLIDING TUB TRANSFER BENCH 14.2 IN .,121.46,...,None,None,None,None,2025-10-10 20:29:17.877158+00:00,2026-01-17,66.0,15880,NaT,NaN


## 6. Optional — all bins in one dict

In [ ]:
# dfs = run_extract("all")
# {k: len(v) for k, v in dfs.items()}

## 7. Approval gate

- [ ] Row counts plausible (match `ai_scripts/output/unified_bin*_1.csv` if you ran the CLI review)
- [ ] Samples look sane
- [ ] vendor_name / manifest null rates acceptable

Then run the next cell to pickle.

## 8. Save pickles (after approval)

In [14]:
CACHE_DIR
save_pickle(df1, bin_name="bin1")
save_pickle(df2, bin_name="bin2")
save_pickle(df3, bin_name="bin3")

WindowsPath('E:/ecothrift-dashboard/workspace/notebooks/category-research/cache/extract_bin3.pkl')

## Phase 4: Taxonomy estimation (manifest proxy)

Load **approved pickles only** (no `run_extract`). First review the **manifest → proposed** table (`ambiguous` = best guess you may want to change). Then see each bin’s share of the 19 canonical categories from [`taxonomy_v1.example.json`](taxonomy_v1.example.json). Uses `manifest_category` as a proxy—not final AI categorization.

In [3]:
import importlib
import cr
importlib.reload(cr)
from IPython.display import display

from cr import (
    load_extract_pickle,
    manifest_mapping_audit_table,
    proposed_distribution,
)

df1 = load_extract_pickle("bin1")
df2 = load_extract_pickle("bin2")
df3 = load_extract_pickle("bin3")

print("MANIFEST → proposed (review ambiguous flags before trusting percentages)")
display(manifest_mapping_audit_table())

for title, d in (
    ("Bin 1 — proposed distribution (19 rows, zeros included)", df1),
    ("Bin 2 — proposed distribution (19 rows, zeros included)", df2),
    ("Bin 3 — proposed distribution (19 rows, zeros included)", df3),
):
    print()
    print(title)
    display(proposed_distribution(d))

MANIFEST → proposed (review ambiguous flags before trusting percentages)


,manifest_label,proposed_category,ambiguous
0,Accent Chairs,Home décor & lighting,False
1,Accent Furniture,Home décor & lighting,False
2,ACCESSORIES,Apparel & accessories,False
3,Accessories,Apparel & accessories,False
4,Action Figures and Collectibles,Toys & games,False
...,...,...,...
439,WOMENS_SHOES,Apparel & accessories,False
440,Woodwind and Brass,Electronics,True
441,Woodworking,Tools & hardware,False
442,Youth Bedroom,Baby & kids,False



Bin 1 — proposed distribution (19 rows, zeros included)


,proposed_category,item_count,percentage_of_bin
0,Mixed lots & uncategorized,9006,16.933346
1,Kitchen & dining,7553,14.201373
2,Home décor & lighting,5807,10.918492
3,Toys & games,5621,10.568769
4,Household & cleaning,4947,9.301495
5,Sports & outdoors,3631,6.827113
6,Tools & hardware,3261,6.131428
7,"Health, beauty & personal care",2126,3.997368
8,Baby & kids,1916,3.602520
9,Outdoor & patio furniture,1481,2.784620



Bin 2 — proposed distribution (19 rows, zeros included)


,proposed_category,item_count,percentage_of_bin
0,Mixed lots & uncategorized,1499,20.514575
1,Kitchen & dining,1365,18.680717
2,Sports & outdoors,734,10.045162
3,Home décor & lighting,592,8.101820
4,Toys & games,530,7.253319
5,Household & cleaning,459,6.281648
6,Tools & hardware,457,6.254277
7,Baby & kids,334,4.570959
8,"Health, beauty & personal care",312,4.269878
9,Bedding & bath,300,4.105652



Bin 3 — proposed distribution (19 rows, zeros included)


,proposed_category,item_count,percentage_of_bin
0,Mixed lots & uncategorized,1845,21.493476
1,Kitchen & dining,1508,17.567568
2,Home décor & lighting,1262,14.701771
3,Sports & outdoors,733,8.539143
4,Household & cleaning,530,6.174278
5,Tools & hardware,486,5.661696
6,Bedding & bath,317,3.692917
7,Apparel & accessories,306,3.564772
8,Electronics,276,3.215284
9,Toys & games,274,3.191985


In [4]:
import os
out = r"E:\ecothrift-dashboard\workspace\notebooks\category-research\ai_scripts\output"
os.makedirs(out, exist_ok=True)

manifest_mapping_audit_table().to_csv(os.path.join(out, "phase4_manifest_mapping_audit.csv"), index=False)
proposed_distribution(df1).to_csv(os.path.join(out, "phase4_bin1_distribution.csv"), index=False)
proposed_distribution(df2).to_csv(os.path.join(out, "phase4_bin2_distribution.csv"), index=False)
proposed_distribution(df3).to_csv(os.path.join(out, "phase4_bin3_distribution.csv"), index=False)
print("Exported to ai_scripts/output/phase4_*.csv")

Exported to ai_scripts/output/phase4_*.csv


## 10. Next phase (stub)

- Build canonical categories from Bin 1 distribution (`taxonomy_v1.example.json` in this folder).
- Export a CSV compatible with `categorize_category_bins` or extend `TAXONOMY_INPUT_FIELDS` to match unified columns.
- See `cr` package, `scripts/sql/unified_bin*_public.sql`, and `ai_scripts/ai_execute_sql.py` for agent CSV review.